# PCA Multi-Method: 10×10-fold CV

**Seed management :**
- 10 independent MD5-derived seeds
- For each seed: create fold splits, then for each decomp×clf block, reset PRNG to the seed
- Each model gets a unique `random_state` drawn from the seeded PRNG
- Mersenne Twister (MT19937) throughout — no PCG
- Models across seeds are truly independent
- Models across decomp×clf blocks within same seed are independent

**Auto-resumes from Google Drive if disconnected.**

In [ ]:
# ============================================================
# 0. Mount Drive + install
# ============================================================

import os
BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(''), '..'))
RESULTS_DIR = os.path.join(BASE_DIR, 'results', 'cv_results_v15')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.chdir(BASE_DIR)
print('CWD:', os.getcwd())
print('Results dir:', RESULTS_DIR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.4/610.4 kB 15.1 MB/s eta 0:00:00
Mounted at /content/drive
CWD: /content/drive/My Drive/CRISPR Hirsfeld
Results dir: /content/drive/My Drive/CRISPR Hirsfeld/cv_results_v15


In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import numpy as np
import pandas as pd
import hashlib
import json
import re
import warnings
warnings.filterwarnings('ignore')
from time import time

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.decomposition import PCA, KernelPCA, SparsePCA, TruncatedSVD, FactorAnalysis
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

print('All imports OK')

All imports OK


In [ ]:
# ============================================================
# 2. MCEN (Delgado & Nunez-Gonzalez 2019)
#
# Two bugs fixed vs all previous versions:
#
#   Bug 1 — Mj denominator for per-class probabilities:
#     OLD: Mj = (col_sum_j - C[j,j]) + (row_sum_j - C[j,j])
#               = FP_j + FN_j   (subtracts diagonal TWICE)
#     FIX: Mj = col_sum_j + row_sum_j - C[j,j]
#               = TP_j + FP_j + FN_j   (subtracts diagonal ONCE)
#
#   Bug 2 — Weight denominator:
#     OLD: wj = Mj / (2 * off_diag)       [alpha=1 for K>2, alpha=0 for K=2]
#     FIX: wj = Mj / (2*n - 0.5*sum(TP))  [Delgado: alpha=0.5 always]
#
#   Entropy and normalisation (log2(2*(N-1))) are unchanged and correct.
#
# Validated against test reference values:
#   [[5,1],[1,5]]         → 0.5910  ✓
#   [[3,0],[6,3]]         → 0.3343  ✓
#   [[30441,75],[43,88]]  → 0.02746 ✓
# ============================================================
def compute_mcen(y_true, y_pred):
    """
    Overall MCEN — Delgado & Nunez-Gonzalez (2019), PLoS ONE 14(1): e0210264.
    Fixes applied vs PyCM:
      - Mj = TP_j + FP_j + FN_j  (not FP_j + FN_j)
      - weight denominator = 2*n - 0.5*sum(TP)  (Delgado alpha=0.5)
    """
    from sklearn.metrics import confusion_matrix as sk_cm
    C = sk_cm(y_true, y_pred).astype(float)
    N = C.shape[0]
    n_total = C.sum()
    tp_sum = np.trace(C)
    if n_total == 0 or N < 2 or tp_sum == n_total:
        return 0.0
    denom_w = 2 * n_total - 0.5 * tp_sum   # Delgado alpha=0.5
    if denom_w <= 0:
        return 0.0
    max_Hj = np.log2(2 * (N - 1)) if N > 2 else 1.0
    mcen = 0.0
    for j in range(N):
        Mj = C[:, j].sum() + C[j, :].sum() - C[j, j]  # TP+FP+FN for class j
        if Mj == 0:
            continue
        Hj = 0.0
        for k in range(N):
            if k == j:
                continue
            for val in (C[k, j], C[j, k]):
                p = val / Mj
                if p > 0:
                    Hj -= p * np.log2(p)
        wj = Mj / denom_w
        mcen += wj * (Hj / max_Hj if max_Hj > 0 else 0.0)
    return float(mcen)

print('compute_mcen defined (Delgado 2019, alpha=0.5, both bugs fixed).')


compute_mcen defined (Delgado 2019, alpha=0.5, both bugs fixed).


In [ ]:
# ============================================================
# 2b. MCEN Validation — MUST ALL PASS before running CV loop
#
# ============================================================
def _run_mcen_validation():
    import numpy as _np

    def _mcen(arr):
        C = _np.array(arr, dtype=float)
        N = C.shape[0]
        n = C.sum(); tp = _np.trace(C)
        if n == 0 or N < 2 or tp == n: return 0.0
        dw = 2*n - 0.5*tp
        if dw <= 0: return 0.0
        mH = _np.log2(2*(N-1)) if N > 2 else 1.0
        out = 0.0
        for j in range(N):
            Mj = C[:,j].sum() + C[j,:].sum() - C[j,j]
            if Mj == 0: continue
            Hj = 0.0
            for k in range(N):
                if k == j: continue
                for v in (C[k,j], C[j,k]):
                    p = v/Mj
                    if p > 0: Hj -= p*_np.log2(p)
            out += (Mj/dw) * (Hj/mH if mH > 0 else 0.0)
        return float(out)

    tests = [
        # (description,                    matrix,                          expected,  tol)
        (" M1 [[5,1],[1,5]]",          [[5,1],[1,5]],                   0.591,    5e-4),
        (" M2 [[3,0],[6,3]]",          [[3,0],[6,3]],                   0.3343,   5e-4),
        (" M3 [[30441,75],[43,88]]",   [[30441,75],[43,88]],            0.02746,  5e-4),
        ("Perfect 3-class (expect 0.0)",   [[10,0,0],[0,8,0],[0,0,12]],    0.0,      1e-10),
        ("Uniform  3-class (expect 1.0)",  [[0,5,5],[5,0,5],[5,5,0]],      1.0,      1e-10),
        ("Perfect 15-class (expect 0.0)",  _np.diag([10]*15).tolist(),     0.0,      1e-10),
    ]

    import numpy as _np
    all_pass = True
    for desc, mat, exp, tol in tests:
        val = _mcen(mat)
        ok = abs(val - exp) < tol
        if not ok: all_pass = False
        print(f"{'PASS' if ok else 'FAIL'}  {desc:<42}  got={val:.6f}  exp={exp}")

    print()
    if all_pass:
        print('All validation tests PASSED. Safe to run CV loop.')
    else:
        print('VALIDATION FAILED — do not run CV loop until all tests pass.')

_run_mcen_validation()


PASS  Prof M1 [[5,1],[1,5]]                       got=0.591022  exp=0.591
PASS  Prof M2 [[3,0],[6,3]]                       got=0.334264  exp=0.3343
PASS  Prof M3 [[30441,75],[43,88]]                got=0.027464  exp=0.02746
PASS  Perfect 3-class (expect 0.0)                got=0.000000  exp=0.0
PASS  Uniform  3-class (expect 1.0)               got=1.000000  exp=1.0
PASS  Perfect 15-class (expect 0.0)               got=0.000000  exp=0.0

All validation tests PASSED. Safe to run CV loop.


In [ ]:
# ============================================================
# 3. Generate 10 independent seeds via MD5 hashing
# ============================================================
N_SEEDS = 10

def md5_seed(label):
    h = hashlib.md5(label.encode('utf-8')).hexdigest()
    return int(h, 16) % (2**31)

SEEDS = [md5_seed(f'crispr_induce_seq_experiment_{i}') for i in range(N_SEEDS)]
print(f'{N_SEEDS} MD5-derived seeds:')
for i, s in enumerate(SEEDS):
    print(f'  Seed {i}: {s}')

10 MD5-derived seeds:
  Seed 0: 1107689332
  Seed 1: 174069893
  Seed 2: 1290585803
  Seed 3: 1872017728
  Seed 4: 2053906564
  Seed 5: 1056733124
  Seed 6: 1178162590
  Seed 7: 204249475
  Seed 8: 55917504
  Seed 9: 1710097361


In [ ]:
# ============================================================
# 4. Load + preprocess data
# ============================================================
df = pd.read_excel('excel full.xlsx')
print('Raw shape:', df.shape)

def pick(cols, candidates):
    for c in candidates:
        if c in cols: return c
    return None

cols = df.columns.tolist()
CONTIG_COL = pick(cols, ['contig','contg','chrom','chromosome','chr'])
SEQ_COL = pick(cols, ['exp_flanking_sequence','flanking_sequence','flank_seq','sequence','seq'])
COORD_START = pick(cols, ['coord_start','cas_coord_start','start'])
COORD_END = pick(cols, ['coord_end','cas_coord_end','end'])
STRAND_COL = pick(cols, ['strand','Strand'])

TOPK = 15
topk = df[CONTIG_COL].astype(str).value_counts().head(TOPK).index
df_work = df[df[CONTIG_COL].astype(str).isin(topk)].copy()

def gc_content(seq):
    seq = str(seq).upper(); v = [b for b in seq if b in 'ACGT']
    return sum(b in 'GC' for b in v) / len(v) if v else 0.0
def gc_skew(seq):
    seq = str(seq).upper(); g, c = seq.count('G'), seq.count('C')
    return (g - c) / (g + c) if (g + c) else 0.0
def at_skew(seq):
    seq = str(seq).upper(); a, t = seq.count('A'), seq.count('T')
    return (a - t) / (a + t) if (a + t) else 0.0
def shannon_entropy(seq):
    seq = str(seq).upper()
    counts = np.array([seq.count(b) for b in 'ACGT'], dtype=np.float32)
    total = counts.sum()
    if total <= 0: return 0.0
    p = counts / total; p = p[p > 0]
    return float(-(p * np.log2(p)).sum())
def max_homopolymer(seq):
    seq = str(seq).upper(); best = run = 0; prev = None
    for b in seq:
        if b not in 'ACGT': run = 0; prev = None; continue
        run = run + 1 if b == prev else 1; prev = b; best = max(best, run)
    return best

df_work['seq_len'] = df_work[SEQ_COL].astype(str).str.len()
df_work['gc_content'] = df_work[SEQ_COL].apply(gc_content)
df_work['gc_skew'] = df_work[SEQ_COL].apply(gc_skew)
df_work['at_skew'] = df_work[SEQ_COL].apply(at_skew)
df_work['entropy'] = df_work[SEQ_COL].apply(shannon_entropy)
df_work['max_homopolymer'] = df_work[SEQ_COL].apply(max_homopolymer)
if COORD_START and COORD_END:
    df_work[COORD_START] = pd.to_numeric(df_work[COORD_START], errors='coerce')
    df_work[COORD_END] = pd.to_numeric(df_work[COORD_END], errors='coerce')
    df_work['dsb_span'] = (df_work[COORD_END] - df_work[COORD_START]).abs()
    df_work['midpoint'] = (df_work[COORD_END] + df_work[COORD_START]) / 2.0
if STRAND_COL:
    s = df_work[STRAND_COL].astype(str).str.strip().str.lower()
    df_work['strand_num'] = np.where(s.isin(['+','plus','1','forward','fwd']), 1,
                            np.where(s.isin(['-','minus','-1','reverse','rev']), -1, 0))

engineered = ['seq_len','gc_content','gc_skew','at_skew','entropy','max_homopolymer']
engineered += [c for c in ['dsb_span','midpoint','strand_num'] if c in df_work.columns]
coord_raw = [c for c in [COORD_START, COORD_END] if c is not None]
numeric_cols = df_work.select_dtypes(include=[np.number]).columns.tolist()
exclude = {SEQ_COL, CONTIG_COL}
exclude |= {c for c in numeric_cols if re.fullmatch(r'(id|ID|index|row_id|site_id)', str(c))}
pca_cols = list(dict.fromkeys(engineered + coord_raw + numeric_cols))
pca_cols = [c for c in pca_cols if c not in exclude]
X = df_work[pca_cols].copy()
na_frac = X.isna().mean()
drop_cols = na_frac[na_frac > 0.60].index.tolist()
if drop_cols: X = X.drop(columns=drop_cols)
X = X.fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

le = LabelEncoder()
y = le.fit_transform(df_work[CONTIG_COL].astype(str).values)
class_names = le.classes_

print(f'Data ready: {X_scaled.shape[0]} samples, {X_scaled.shape[1]} features, {len(class_names)} classes')

Raw shape: (9877, 38)
Data ready: 7863 samples, 34 features, 15 classes


In [ ]:
# ============================================================
# 5. Define methods + seed-aware constructors
# ============================================================
N_COMP = 16
N_FOLDS = 10

# These constructors accept a random_state parameter
# so each call gets a DIFFERENT seed from the PRNG

def make_decomp(name, rs):
    """Create decomposition with given random_state."""
    if name == 'Standard PCA':
        return PCA(n_components=N_COMP)
    elif name == 'Kernel PCA (RBF)':
        return KernelPCA(n_components=N_COMP, kernel='rbf', gamma=None, random_state=rs)
    elif name == 'Sparse PCA':
        return SparsePCA(n_components=N_COMP, alpha=1.0, ridge_alpha=0.01,
                         max_iter=100, random_state=rs, n_jobs=-1)
    elif name == 'Truncated SVD':
        return TruncatedSVD(n_components=N_COMP, random_state=rs)
    elif name == 'Factor Analysis':
        return FactorAnalysis(n_components=N_COMP, random_state=rs)

def make_clf(name, rs):
    """Create classifier with given random_state."""
    if name == 'Decision Tree':
        return DecisionTreeClassifier(max_depth=None, random_state=rs)
    elif name == 'Random Forest':
        return RandomForestClassifier(n_estimators=300, max_depth=None,
                                      min_samples_leaf=3, random_state=rs, n_jobs=-1)
    elif name == 'Extra Trees':
        return ExtraTreesClassifier(n_estimators=300, max_depth=None,
                                    min_samples_leaf=3, random_state=rs, n_jobs=-1)

decomp_names = ['Standard PCA', 'Kernel PCA (RBF)', 'Sparse PCA', 'Truncated SVD', 'Factor Analysis']
clf_names = ['Decision Tree', 'Random Forest', 'Extra Trees']

print(f'{len(decomp_names)} decompositions x {len(clf_names)} classifiers x {N_SEEDS} seeds x {N_FOLDS} folds')
print(f'Total experiments: {len(decomp_names) * len(clf_names) * N_SEEDS * N_FOLDS}')
print(f'\nSeed management: Option B (global block-repeated)')
print(f'PRNG: Mersenne Twister (MT19937) via np.random.RandomState')

5 decompositions x 3 classifiers x 10 seeds x 10 folds
Total experiments: 1500

Seed management: Option B (global block-repeated)
PRNG: Mersenne Twister (MT19937) via np.random.RandomState


In [ ]:
# ============================================================
# 6. MAIN LOOP: 10 seeds x 10-fold CV
#    Seed management: Option B (global block-repeated)
#
#    Pseudocode:
#    for seed in MD5_seeds:                    # 10 independent measurements
#        rng_split = RandomState(seed)
#        fold_indices = StratifiedKFold(random_state=rng_split)
#        for each decomp_name:
#            for each clf_name:
#                rng_block = RandomState(seed)  # RESET for each block
#                for each fold:
#                    fold_rs = rng_block.randint()
#                    decomp = make_decomp(decomp_name, random_state=fold_rs)
#                    clf = make_clf(clf_name, random_state=fold_rs)
#                    train, evaluate, collect metrics
#                average fold metrics -> ONE measurement
#
#    This ensures:
#    - Different seeds -> different fold splits AND different model randomness
#    - Different decomp×clf blocks within same seed are independent
#    - No fixed random_state=42 anywhere
# ============================================================
t_total_start = time()

for seed_idx, seed in enumerate(SEEDS):
    result_file = os.path.join(RESULTS_DIR, f'seed_{seed_idx}_{seed}.json')

    # Auto-resume: skip completed seeds
    if os.path.exists(result_file):
        print(f'Seed {seed_idx}/{N_SEEDS} ({seed}) — ALREADY DONE, skipping.')
        continue

    print(f'\n===== Seed {seed_idx}/{N_SEEDS} ({seed}) =====')
    t_seed_start = time()

    # Step 1: Create fold splits using this seed
    rng_split = np.random.RandomState(seed)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rng_split)
    fold_indices = list(skf.split(X_scaled, y))

    seed_results = {}
    combo_count = 0
    total_combos = len(decomp_names) * len(clf_names)

    for d_name in decomp_names:
        for c_name in clf_names:
            combo_count += 1

            # Step 2: RESET PRNG for this decomp×clf block
            # This makes each block independent of other blocks
            rng_block = np.random.RandomState(seed)

            fold_accs, fold_f1s, fold_mccs, fold_mcens = [], [], [], []

            for train_idx, test_idx in fold_indices:
                # Step 3: Draw a unique random_state for this fold's models
                fold_rs = int(rng_block.randint(0, 2**31))

                # Decomposition with fold-specific random_state
                decomp = make_decomp(d_name, rs=fold_rs)
                X_tr_d = decomp.fit_transform(X_scaled[train_idx])
                X_te_d = decomp.transform(X_scaled[test_idx])

                # Classifier with fold-specific random_state
                clf = make_clf(c_name, rs=fold_rs)
                clf.fit(X_tr_d, y[train_idx])
                y_pred = clf.predict(X_te_d)
                y_true = y[test_idx]

                fold_accs.append(accuracy_score(y_true, y_pred))
                fold_f1s.append(f1_score(y_true, y_pred, average='macro'))
                fold_mccs.append(matthews_corrcoef(y_true, y_pred))
                fold_mcens.append(compute_mcen(y_true, y_pred))

            # Average across folds -> ONE measurement per seed
            key = f'{d_name}|{c_name}'
            seed_results[key] = {
                'Accuracy': float(np.mean(fold_accs)),
                'Macro-F1': float(np.mean(fold_f1s)),
                'MCC': float(np.mean(fold_mccs)),
                'MCEN': float(np.mean(fold_mcens)),
            }
            elapsed = time() - t_seed_start
            print(f'  [{combo_count:2d}/{total_combos}] {d_name:20s} + {c_name:15s}  '
                  f'Acc={np.mean(fold_accs):.4f}  ({elapsed:.0f}s)')

    # Save to Drive
    with open(result_file, 'w') as f:
        json.dump({
            'seed': seed, 'seed_idx': seed_idx,
            'seed_management': 'Option B: global block-repeated',
            'prng': 'Mersenne Twister (MT19937) via np.random.RandomState',
            'results': seed_results
        }, f, indent=2)
    print(f'  Seed {seed_idx} saved. ({time()-t_seed_start:.0f}s)')

print(f'\n=== ALL SEEDS COMPLETE ({time()-t_total_start:.0f}s total) ===')


===== Seed 0/10 (1107689332) =====
  [ 1/15] Standard PCA         + Decision Tree    Acc=0.2411  (11s)
  [ 2/15] Standard PCA         + Random Forest    Acc=0.2886  (186s)
  [ 3/15] Standard PCA         + Extra Trees      Acc=0.2907  (216s)
  [ 4/15] Kernel PCA (RBF)     + Decision Tree    Acc=0.1910  (825s)
  [ 5/15] Kernel PCA (RBF)     + Random Forest    Acc=0.2635  (1559s)
  [ 6/15] Kernel PCA (RBF)     + Extra Trees      Acc=0.2616  (2185s)
  [ 7/15] Sparse PCA           + Decision Tree    Acc=0.3672  (2285s)
  [ 8/15] Sparse PCA           + Random Forest    Acc=0.3541  (2535s)
  [ 9/15] Sparse PCA           + Extra Trees      Acc=0.3606  (2657s)
  [10/15] Truncated SVD        + Decision Tree    Acc=0.2396  (2663s)
  [11/15] Truncated SVD        + Random Forest    Acc=0.2881  (2825s)
  [12/15] Truncated SVD        + Extra Trees      Acc=0.2900  (2855s)
  [13/15] Factor Analysis      + Decision Tree    Acc=0.3502  (2958s)
  [14/15] Factor Analysis      + Random Forest    Acc=0.350

In [ ]:
# ============================================================
# 7. Load all seed results + build summary
# ============================================================
all_seed_data = []
for seed_idx in range(N_SEEDS):
    seed = SEEDS[seed_idx]
    result_file = os.path.join(RESULTS_DIR, f'seed_{seed_idx}_{seed}.json')
    if not os.path.exists(result_file):
        print(f'WARNING: seed {seed_idx} missing!')
        continue
    with open(result_file) as f:
        data = json.load(f)
    for combo_key, metrics in data['results'].items():
        decomp, clf = combo_key.split('|')
        all_seed_data.append({
            'Seed_idx': seed_idx, 'Seed': seed,
            'Decomposition': decomp, 'Classifier': clf,
            **metrics
        })

df_all = pd.DataFrame(all_seed_data)
print(f'Loaded {len(df_all)} rows ({df_all["Seed_idx"].nunique()} seeds)')

summary_rows = []
for decomp in decomp_names:
    for clf in clf_names:
        mask = (df_all['Decomposition'] == decomp) & (df_all['Classifier'] == clf)
        sub = df_all[mask]
        row = {'Decomposition': decomp, 'Classifier': clf}
        for metric in ['Accuracy', 'Macro-F1', 'MCC', 'MCEN']:
            vals = sub[metric].values
            row[f'{metric}_mean'] = np.mean(vals)
            row[f'{metric}_std'] = np.std(vals)
            row[f'{metric}_str'] = f'{np.mean(vals):.4f}\u00B1{np.std(vals):.4f}'
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print('\n=== RESULTS (mean \u00B1 std across 10 independent repetitions) ===')
display(df_summary[['Decomposition','Classifier','Accuracy_str','Macro-F1_str','MCC_str','MCEN_str']])

Loaded 150 rows (10 seeds)

=== RESULTS (mean ± std across 10 independent repetitions) ===


,Decomposition,Classifier,Accuracy_str,Macro-F1_str,MCC_str,MCEN_str
0,Standard PCA,Decision Tree,0.2362±0.0043,0.2181±0.0050,0.1781±0.0047,0.7885±0.0042
1,Standard PCA,Random Forest,0.2912±0.0022,0.2582±0.0028,0.2372±0.0025,0.6986±0.0017
2,Standard PCA,Extra Trees,0.2930±0.0023,0.2544±0.0028,0.2390±0.0026,0.6870±0.0020
3,Kernel PCA (RBF),Decision Tree,0.1938±0.0020,0.1775±0.0020,0.1324±0.0021,0.8183±0.0014
4,Kernel PCA (RBF),Random Forest,0.2664±0.0032,0.2312±0.0038,0.2100±0.0035,0.7125±0.0031
5,Kernel PCA (RBF),Extra Trees,0.2624±0.0024,0.2225±0.0021,0.2058±0.0027,0.7056±0.0027
6,Sparse PCA,Decision Tree,0.3707±0.0051,0.3522±0.0045,0.3228±0.0054,0.6869±0.0042
7,Sparse PCA,Random Forest,0.3560±0.0044,0.3277±0.0045,0.3084±0.0047,0.6521±0.0042
8,Sparse PCA,Extra Trees,0.3584±0.0029,0.3255±0.0041,0.3110±0.0033,0.6429±0.0026
9,Truncated SVD,Decision Tree,0.2347±0.0034,0.2167±0.0039,0.1764±0.0036,0.7895±0.0037


In [ ]:
# ============================================================
# 8. Pivot tables
# ============================================================
for metric in ['Accuracy', 'Macro-F1', 'MCC', 'MCEN']:
    print(f'\n=== {metric} PIVOT ===')
    pivot = df_summary.pivot(index='Decomposition', columns='Classifier', values=f'{metric}_str')
    display(pivot)


=== Accuracy PIVOT ===


Classifier,Decision Tree,Extra Trees,Random Forest
Decomposition,,,
Factor Analysis,0.3531±0.0035,0.3636±0.0036,0.3544±0.0031
Kernel PCA (RBF),0.1938±0.0020,0.2624±0.0024,0.2664±0.0032
Sparse PCA,0.3707±0.0051,0.3584±0.0029,0.3560±0.0044
Standard PCA,0.2362±0.0043,0.2930±0.0023,0.2912±0.0022
Truncated SVD,0.2347±0.0034,0.2920±0.0015,0.2914±0.0025



=== Macro-F1 PIVOT ===


Classifier,Decision Tree,Extra Trees,Random Forest
Decomposition,,,
Factor Analysis,0.3342±0.0041,0.3261±0.0039,0.3203±0.0032
Kernel PCA (RBF),0.1775±0.0020,0.2225±0.0021,0.2312±0.0038
Sparse PCA,0.3522±0.0045,0.3255±0.0041,0.3277±0.0045
Standard PCA,0.2181±0.0050,0.2544±0.0028,0.2582±0.0028
Truncated SVD,0.2167±0.0039,0.2534±0.0023,0.2583±0.0024



=== MCC PIVOT ===


Classifier,Decision Tree,Extra Trees,Random Forest
Decomposition,,,
Factor Analysis,0.3039±0.0038,0.3171±0.0039,0.3071±0.0035
Kernel PCA (RBF),0.1324±0.0021,0.2058±0.0027,0.2100±0.0035
Sparse PCA,0.3228±0.0054,0.3110±0.0033,0.3084±0.0047
Standard PCA,0.1781±0.0047,0.2390±0.0026,0.2372±0.0025
Truncated SVD,0.1764±0.0036,0.2379±0.0016,0.2375±0.0027



=== MCEN PIVOT ===


Classifier,Decision Tree,Extra Trees,Random Forest
Decomposition,,,
Factor Analysis,0.7001±0.0026,0.6288±0.0028,0.6475±0.0027
Kernel PCA (RBF),0.8183±0.0014,0.7056±0.0027,0.7125±0.0031
Sparse PCA,0.6869±0.0042,0.6429±0.0026,0.6521±0.0042
Standard PCA,0.7885±0.0042,0.6870±0.0020,0.6986±0.0017
Truncated SVD,0.7895±0.0037,0.6877±0.0023,0.6986±0.0021


In [ ]:
# ============================================================
# 9. Friedman ANOVA
# ============================================================
from scipy.stats import friedmanchisquare

print('=== FRIEDMAN ANOVA (per classifier, across decompositions) ===')
print('Each of 10 repetitions provides one averaged measurement.\n')

friedman_results = []
for metric in ['Accuracy', 'Macro-F1', 'MCC']:
    print(f'--- {metric} ---')
    for c_name in clf_names:
        arrays = []
        for d_name in decomp_names:
            mask = (df_all['Decomposition'] == d_name) & (df_all['Classifier'] == c_name)
            vals = df_all[mask].sort_values('Seed_idx')[metric].values
            arrays.append(vals)
        stat, p = friedmanchisquare(*arrays)
        df_f = len(decomp_names) - 1
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {c_name:15s}: \u03C7\u00B2({df_f}) = {stat:.2f}, p = {p:.4e} {sig}')
        friedman_results.append({'Metric': metric, 'Classifier': c_name,
                                  'chi2': stat, 'df': df_f, 'p': p, 'sig': sig})
    print()

df_friedman = pd.DataFrame(friedman_results)
df_friedman.to_csv(os.path.join(RESULTS_DIR, 'friedman_results.csv'), index=False)

=== FRIEDMAN ANOVA (per classifier, across decompositions) ===
Each of 10 repetitions provides one averaged measurement.

--- Accuracy ---
  Decision Tree  : χ²(4) = 38.72, p = 7.9586e-08 ***
  Random Forest  : χ²(4) = 36.08, p = 2.7861e-07 ***
  Extra Trees    : χ²(4) = 37.44, p = 1.4619e-07 ***

--- Macro-F1 ---
  Decision Tree  : χ²(4) = 38.72, p = 7.9586e-08 ***
  Random Forest  : χ²(4) = 37.36, p = 1.5185e-07 ***
  Extra Trees    : χ²(4) = 36.08, p = 2.7861e-07 ***

--- MCC ---
  Decision Tree  : χ²(4) = 38.72, p = 7.9586e-08 ***
  Random Forest  : χ²(4) = 36.16, p = 2.6825e-07 ***
  Extra Trees    : χ²(4) = 37.44, p = 1.4619e-07 ***



In [ ]:
# ============================================================
# 10. Post-hoc Wilcoxon with Holm correction
# ============================================================
from scipy.stats import wilcoxon
from itertools import combinations

def holm_correction(pvals):
    n = len(pvals)
    sorted_idx = np.argsort(pvals)
    corrected = np.zeros(n)
    for rank, idx in enumerate(sorted_idx):
        corrected[idx] = min(pvals[idx] * (n - rank), 1.0)
    for i in range(1, n):
        corrected[sorted_idx[i]] = max(corrected[sorted_idx[i]], corrected[sorted_idx[i-1]])
    return corrected

print('=== POST-HOC PAIRWISE WILCOXON (Holm-corrected) ===\n')
posthoc_results = []

for metric in ['Accuracy', 'Macro-F1', 'MCC']:
    for c_name in clf_names:
        pairs = list(combinations(range(len(decomp_names)), 2))
        raw_pvals = []
        pair_labels = []
        for i, j in pairs:
            mask_i = (df_all['Decomposition'] == decomp_names[i]) & (df_all['Classifier'] == c_name)
            mask_j = (df_all['Decomposition'] == decomp_names[j]) & (df_all['Classifier'] == c_name)
            vals_i = df_all[mask_i].sort_values('Seed_idx')[metric].values
            vals_j = df_all[mask_j].sort_values('Seed_idx')[metric].values
            try:
                stat, p = wilcoxon(vals_i, vals_j)
            except ValueError:
                p = 1.0
            raw_pvals.append(p)
            pair_labels.append(f'{decomp_names[i]} vs {decomp_names[j]}')

        corrected = holm_correction(raw_pvals)
        for k, (pi, pj) in enumerate(pairs):
            sig = '***' if corrected[k] < 0.001 else '**' if corrected[k] < 0.01 else '*' if corrected[k] < 0.05 else 'ns'
            posthoc_results.append({
                'Metric': metric, 'Classifier': c_name,
                'Pair': pair_labels[k],
                'p_raw': raw_pvals[k], 'p_holm': corrected[k], 'Sig': sig
            })

df_posthoc = pd.DataFrame(posthoc_results)
sig_count = (df_posthoc['Sig'] != 'ns').sum()
print(f'Significant pairs: {sig_count} / {len(df_posthoc)}')
print('\n--- Accuracy ---')
display(df_posthoc[df_posthoc['Metric']=='Accuracy'][['Classifier','Pair','p_raw','p_holm','Sig']].round(4))

df_posthoc.to_csv(os.path.join(RESULTS_DIR, 'posthoc_wilcoxon.csv'), index=False)

=== POST-HOC PAIRWISE WILCOXON (Holm-corrected) ===

Significant pairs: 78 / 90

--- Accuracy ---


,Classifier,Pair,p_raw,p_holm,Sig
0,Decision Tree,Standard PCA vs Kernel PCA (RBF),0.0020,0.0195,*
1,Decision Tree,Standard PCA vs Sparse PCA,0.0020,0.0195,*
2,Decision Tree,Standard PCA vs Truncated SVD,0.0645,0.0645,ns
3,Decision Tree,Standard PCA vs Factor Analysis,0.0020,0.0195,*
4,Decision Tree,Kernel PCA (RBF) vs Sparse PCA,0.0020,0.0195,*
5,Decision Tree,Kernel PCA (RBF) vs Truncated SVD,0.0020,0.0195,*
6,Decision Tree,Kernel PCA (RBF) vs Factor Analysis,0.0020,0.0195,*
7,Decision Tree,Sparse PCA vs Truncated SVD,0.0020,0.0195,*
8,Decision Tree,Sparse PCA vs Factor Analysis,0.0020,0.0195,*
9,Decision Tree,Truncated SVD vs Factor Analysis,0.0020,0.0195,*


In [ ]:
# ============================================================
# 11. Save final summary
# ============================================================
df_summary.to_csv(os.path.join(RESULTS_DIR, 'summary_10seeds.csv'), index=False)
df_all.to_csv(os.path.join(RESULTS_DIR, 'all_seed_results.csv'), index=False)

print('=== FINAL SUMMARY ===')
print(f'Seeds: {N_SEEDS} (MD5-derived, Mersenne Twister MT19937)')
print(f'Folds per seed: {N_FOLDS}')
print(f'Seed management: Option B (global block-repeated)')
print(f'Total experiments: {N_SEEDS * N_FOLDS * len(decomp_names) * len(clf_names)}')

best_idx = df_summary['Macro-F1_mean'].idxmax()
best = df_summary.loc[best_idx]
print(f'\nBest combo: {best["Decomposition"]} + {best["Classifier"]}')
print(f'  Accuracy: {best["Accuracy_str"]}')
print(f'  Macro-F1: {best["Macro-F1_str"]}')
print(f'  MCC:      {best["MCC_str"]}')
print(f'\nAll results saved to: {RESULTS_DIR}')

=== FINAL SUMMARY ===
Seeds: 10 (MD5-derived, Mersenne Twister MT19937)
Folds per seed: 10
Seed management: Option B (global block-repeated)
Total experiments: 1500

Best combo: Sparse PCA + Decision Tree
  Accuracy: 0.3707±0.0051
  Macro-F1: 0.3522±0.0045
  MCC:      0.3228±0.0054

All results saved to: /content/drive/My Drive/CRISPR Hirsfeld/cv_results_v15
